In [1]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

PHX = Path("/home/soehn/Dataset/CV/PHOENIX-2014-T-release-v3/PHOENIX-2014-T")
ANN = PHX / "annotations" / "manual"
FRM = PHX / "features" / "fullFrame-210x260px"
W   = Path.home() / "work/signlanguage"

for p in [PHX, ANN, FRM, W]:
    print("%-6s %s" % ("OK" if p.exists() else "MISSING", p))

OK     /home/soehn/Dataset/CV/PHOENIX-2014-T-release-v3/PHOENIX-2014-T
OK     /home/soehn/Dataset/CV/PHOENIX-2014-T-release-v3/PHOENIX-2014-T/annotations/manual
OK     /home/soehn/Dataset/CV/PHOENIX-2014-T-release-v3/PHOENIX-2014-T/features/fullFrame-210x260px
OK     /home/soehn/work/signlanguage


In [2]:
phx = {s: pd.read_csv(ANN / ("PHOENIX-2014-T.%s.corpus.csv" % s), sep="|")
       for s in ["train", "dev", "test"]}

for s, d in phx.items():
    print("%-5s rows %5d | columns %s" % (s, len(d), list(d.columns)))

print("\nmissing values:")
print(phx["train"].isna().sum().to_string())

print("\nfirst 3 training rows:")
for _, r in phx["train"].head(3).iterrows():
    print("\n name       :", r["name"])
    print(" speaker    :", r["speaker"])
    print(" glosses    :", r["orth"])
    print(" translation:", r["translation"])

train rows  7096 | columns ['name', 'video', 'start', 'end', 'speaker', 'orth', 'translation']
dev   rows   519 | columns ['name', 'video', 'start', 'end', 'speaker', 'orth', 'translation']
test  rows   642 | columns ['name', 'video', 'start', 'end', 'speaker', 'orth', 'translation']

missing values:
name           0
video          0
start          0
end            0
speaker        0
orth           0
translation    0

first 3 training rows:

 name       : 11August_2010_Wednesday_tagesschau-1
 speaker    : Signer08
 glosses    : JETZT WETTER MORGEN DONNERSTAG ZWOELF FEBRUAR
 translation: und nun die wettervorhersage für morgen donnerstag den zwölften august

 name       : 11August_2010_Wednesday_tagesschau-4
 speaker    : Signer08
 glosses    : ORT REGEN DURCH REGEN KOENNEN UEBERSCHWEMMUNG KOENNEN
 translation: mancherorts regnet es auch länger und ergiebig auch lokale überschwemmungen sind wieder möglich

 name       : 11August_2010_Wednesday_tagesschau-5
 speaker    : Signer08
 glosse

In [3]:
def vocab_stats(sentences, tokenize, label):
    """tokenize: function str -> list of tokens. Returns (stats dict, Counter)."""
    toks = [t for s in sentences for t in tokenize(str(s)) if t.strip()]
    c = Counter(toks)
    n_tok, n_typ = len(toks), len(c)
    once = sum(1 for v in c.values() if v == 1)
    rare = sum(1 for v in c.values() if v < 5)
    cover5 = sum(v for v in c.values() if v >= 5) / n_tok
    return {
        "set": label,
        "sentences": len(sentences),
        "tokens": n_tok,
        "unique words": n_typ,
        "avg times a word appears": round(n_tok / n_typ, 1),
        "words seen once %": round(100 * once / n_typ, 1),
        "words seen <5 times %": round(100 * rare / n_typ, 1),
        "text covered by words seen >=5x %": round(100 * cover5, 1),
        "words per sentence": round(n_tok / len(sentences), 1),
    }, c

ws = lambda s: s.lower().split()      # whitespace tokenizer (German, English)
print("ready")

ready


In [4]:
rows = []
s_de, c_de = vocab_stats(phx["train"]["translation"], ws, "PHOENIX German (train)")
s_gl, c_gl = vocab_stats(phx["train"]["orth"],        ws, "PHOENIX glosses (train)")
rows += [s_de, s_gl]

pd.set_option("display.width", 200)
display(pd.DataFrame(rows).set_index("set").T)

print("\n20 most common German words:")
print(c_de.most_common(20))
print("\n20 most common glosses:")
print(c_gl.most_common(20))

set,PHOENIX German (train),PHOENIX glosses (train)
sentences,7096.0,7096.0
tokens,99081.0,55247.0
unique words,2887.0,1085.0
avg times a word appears,34.3,50.9
words seen once %,37.3,32.7
words seen <5 times %,60.9,56.3
text covered by words seen >=5x %,97.1,98.1
words per sentence,14.0,7.8



20 most common German words:
[('und', 4156), ('im', 2934), ('es', 2852), ('der', 2462), ('am', 2197), ('in', 1938), ('die', 1865), ('auch', 1747), ('grad', 1588), ('bis', 1526), ('den', 1429), ('morgen', 1292), ('nacht', 1042), ('noch', 974), ('an', 962), ('heute', 838), ('regen', 838), ('sich', 825), ('wolken', 823), ('ein', 747)]

20 most common glosses:
[('regen', 2149), ('region', 1883), ('ix', 1505), ('kommen', 1324), ('morgen', 1263), ('nord', 1216), ('sonne', 1140), ('wolke', 1107), ('grad', 1049), ('nacht', 860), ('sued', 835), ('koennen', 814), ('schnee', 767), ('auch', 762), ('mehr', 752), ('bisschen', 751), ('heute', 736), ('bis', 685), ('gewitter', 672), ('wetter', 669)]


In [10]:
from pythainlp.tokenize import word_tokenize
th = lambda s: word_tokenize(s, engine="newmm")

# ---- inputs
sp    = pd.read_csv(W / "splits/split_v2.csv")
en    = pd.read_csv(W / "targets/en_targets.csv")
trs   = sp[sp["split"] == "train"]
tr_en = trs[["clip_id"]].merge(
            en[en["en_ok"].astype(bool)][["clip_id", "text_en"]], on="clip_id")
print("train clips:", len(trs), "| with English:", len(tr_en))

# ---- everything recomputed into a dict (re-running this cell is safe)
STATS, COUNTS = {}, {}

def add(label, sentences, tokenize):
    s, c = vocab_stats(sentences, tokenize, label)
    STATS[label], COUNTS[label] = s, c

add("PHOENIX German (train)",      phx["train"]["translation"], ws)
add("PHOENIX glosses (train)",     phx["train"]["orth"],        ws)
add("TSL English, medical (train)", tr_en["text_en"],           ws)
add("TSL Thai, all (train)",        trs["text_th_final"],       th)
add("TSL Thai, cooking (train)",    trs[trs["batch"] == "batch1"]["text_th_final"], th)
add("TSL Thai, medical (train)",    trs[trs["batch"] == "batch2"]["text_th_final"], th)

table = pd.DataFrame(STATS.values()).set_index("set").T
display(table)

train clips: 2577 | with English: 1772


set,PHOENIX German (train),PHOENIX glosses (train),"TSL English, medical (train)","TSL Thai, all (train)","TSL Thai, cooking (train)","TSL Thai, medical (train)"
sentences,7096.0,7096.0,1772.0,2577.0,805.0,1772.0
tokens,99081.0,55247.0,14184.0,29645.0,15323.0,14322.0
unique words,2887.0,1085.0,2772.0,3415.0,2170.0,1927.0
avg times a word appears,34.3,50.9,5.1,8.7,7.1,7.4
words seen once %,37.3,32.7,57.9,43.7,47.1,44.6
words seen <5 times %,60.9,56.3,85.9,73.0,76.5,75.1
text covered by words seen >=5x %,97.1,98.1,74.4,86.0,82.3,83.1
words per sentence,14.0,7.8,8.0,11.5,19.0,8.1


In [11]:
for n in [1772, 2577]:
    add("PHOENIX German, %d sents" % n,
        phx["train"]["translation"].sample(n, random_state=42), ws)

print("stability — 5 random draws of 1772 PHOENIX sentences:")
for seed in range(5):
    s, _ = vocab_stats(phx["train"]["translation"].sample(1772, random_state=seed), ws, "x")
    print("  unique %4d | avg repeat %5.1f | coverage %.1f%%"
          % (s["unique words"], s["avg times a word appears"],
             s["text covered by words seen >=5x %"]))

cols = ["PHOENIX German (train)", "PHOENIX German, 1772 sents",
        "TSL English, medical (train)", "PHOENIX German, 2577 sents",
        "TSL Thai, all (train)"]
display(pd.DataFrame(STATS.values()).set_index("set").T[cols])

stability — 5 random draws of 1772 PHOENIX sentences:
  unique 1680 | avg repeat  14.7 | coverage 92.9%
  unique 1678 | avg repeat  14.6 | coverage 92.5%
  unique 1716 | avg repeat  14.5 | coverage 92.6%
  unique 1705 | avg repeat  14.6 | coverage 92.8%
  unique 1742 | avg repeat  14.3 | coverage 92.3%


set,PHOENIX German (train),"PHOENIX German, 1772 sents","TSL English, medical (train)","PHOENIX German, 2577 sents","TSL Thai, all (train)"
sentences,7096.0,1772.0,1772.0,2577.0,2577.0
tokens,99081.0,24506.0,14184.0,35653.0,29645.0
unique words,2887.0,1669.0,2772.0,1958.0,3415.0
avg times a word appears,34.3,14.7,5.1,18.2,8.7
words seen once %,37.3,39.6,57.9,39.8,43.7
words seen <5 times %,60.9,64.7,85.9,63.3,73.0
text covered by words seen >=5x %,97.1,92.7,74.4,94.3,86.0
words per sentence,14.0,13.8,8.0,13.8,11.5


In [14]:
def oov(train_sents, test_sents, tokenize, label):
    V = set(t for s in train_sents for t in tokenize(str(s)) if t.strip())
    toks = [t for s in test_sents for t in tokenize(str(s)) if t.strip()]
    miss = [t for t in toks if t not in V]
    sent_hit = sum(1 for s in test_sents
                   if any(t not in V for t in tokenize(str(s)) if t.strip()))
    return {"set": label,
            "train vocabulary": len(V),
            "test tokens": len(toks),
            "unseen tokens %": round(100 * len(miss) / len(toks), 1),
            "unseen distinct words": len(set(miss)),
            "test sentences with >=1 unseen word %": round(100 * sent_hit / len(test_sents), 1)}

# test sets
te_sp = sp[sp["split"] == "test"]
te_en = te_sp[["clip_id"]].merge(
            en[en["en_ok"].astype(bool)][["clip_id", "text_en"]], on="clip_id")

orows = [
    oov(phx["train"]["translation"], phx["test"]["translation"], ws, "PHOENIX German"),
    oov(phx["train"]["orth"],        phx["test"]["orth"],        ws, "PHOENIX glosses"),
    oov(tr_en["text_en"],            te_en["text_en"],           ws, "TSL English (medical)"),
    oov(trs["text_th_final"],        te_sp["text_th_final"],     th, "TSL Thai (all)"),
    oov(trs[trs["batch"]=="batch1"]["text_th_final"],
        te_sp[te_sp["batch"]=="batch1"]["text_th_final"], th, "TSL Thai (cooking)"),
    oov(trs[trs["batch"]=="batch2"]["text_th_final"],
        te_sp[te_sp["batch"]=="batch2"]["text_th_final"], th, "TSL Thai (medical)"),
]
display(pd.DataFrame(orows).set_index("set").T)

set,PHOENIX German,PHOENIX glosses,TSL English (medical),TSL Thai (all),TSL Thai (cooking),TSL Thai (medical)
train vocabulary,2887.0,1085.0,2772.0,3415.0,2170.0,1927.0
test tokens,7816.0,4264.0,1929.0,3768.0,1854.0,1914.0
unseen tokens %,0.8,0.4,10.6,5.0,8.3,4.6
unseen distinct words,59.0,19.0,195.0,178.0,141.0,86.0
test sentences with >=1 unseen word %,7.6,2.5,59.8,36.2,71.8,29.5


In [15]:
V = set(t for s in tr_en["text_en"] for t in ws(str(s)))
miss = sorted({t for s in te_en["text_en"] for t in ws(str(s)) if t not in V})
print("English words in test but never in training: %d" % len(miss))
print(miss[:60])

English words in test but never in training: 195
['(first', '(vertigo)?', '10?', '14', '2021.', 'access', 'accident;', 'active', 'adults?', 'aloe', 'antiviral', 'around.', 'assistance', 'attacks?', 'autopsy', 'awake?', 'bedtime', 'blocks?', 'bookstore?', 'bottle.', 'box?', 'breath?', 'bumps?', 'cake', 'calamine', 'care.', 'changed', 'changing', 'charges', 'checked', 'childhood.', 'circulation?', 'clearly.', 'clots.', 'color?', 'combination', 'communication?', 'community', 'completed', 'compresses', 'concentration?', 'continues?', 'cool,', 'cooperation', 'count.', 'country.', 'cracked', 'cramps.', 'cramps—use', 'depression', 'destination', 'disposal', 'dissolve', 'doctor’s', 'dreams', 'dry,', 'dysmorphic', 'eight', 'emotional', 'enjoy']


In [16]:
import re, unicodedata

PUNCT = re.compile(r"[^\w\s]", flags=re.UNICODE)          # drop . , ? ( ) ’ etc.

def ws_clean(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = s.replace("\u2019", "'").replace("—", " ").replace("–", " ")
    s = PUNCT.sub(" ", s)
    return s.split()

def th_clean(s):
    s = unicodedata.normalize("NFC", str(s))
    s = PUNCT.sub(" ", s)
    return [t for t in word_tokenize(s, engine="newmm") if t.strip()]

# quick check that it does what we expect
print(ws_clean("Are you awake? Use cool, damp compresses — (vertigo)?"))

['are', 'you', 'awake', 'use', 'cool', 'damp', 'compresses', 'vertigo']


In [17]:
orows2 = [
    oov(phx["train"]["translation"], phx["test"]["translation"], ws_clean, "PHOENIX German"),
    oov(phx["train"]["orth"],        phx["test"]["orth"],        ws_clean, "PHOENIX glosses"),
    oov(tr_en["text_en"],            te_en["text_en"],           ws_clean, "TSL English (medical)"),
    oov(trs["text_th_final"],        te_sp["text_th_final"],     th_clean, "TSL Thai (all)"),
    oov(trs[trs["batch"]=="batch1"]["text_th_final"],
        te_sp[te_sp["batch"]=="batch1"]["text_th_final"], th_clean, "TSL Thai (cooking)"),
    oov(trs[trs["batch"]=="batch2"]["text_th_final"],
        te_sp[te_sp["batch"]=="batch2"]["text_th_final"], th_clean, "TSL Thai (medical)"),
]
display(pd.DataFrame(orows2).set_index("set").T)

V = set(t for s in tr_en["text_en"] for t in ws_clean(s))
miss = sorted({t for s in te_en["text_en"] for t in ws_clean(s) if t not in V})
print("\nEnglish words in test but never in training: %d" % len(miss))
print(miss[:60])

set,PHOENIX German,PHOENIX glosses,TSL English (medical),TSL Thai (all),TSL Thai (cooking),TSL Thai (medical)
train vocabulary,2887.0,953.0,2017.0,2318.0,1623.0,1502.0
test tokens,7816.0,4399.0,1967.0,6255.0,2920.0,3335.0
unseen tokens %,0.8,0.3,5.6,1.9,3.2,2.2
unseen distinct words,59.0,12.0,103.0,117.0,89.0,70.0
test sentences with >=1 unseen word %,7.6,1.6,35.9,26.4,53.4,24.8



English words in test but never in training: 103
['14', '2021', 'access', 'active', 'adults', 'aloe', 'antiviral', 'attacks', 'autopsy', 'blocks', 'bookstore', 'bumps', 'calamine', 'changing', 'checked', 'childhood', 'clearly', 'combination', 'communication', 'completed', 'compresses', 'concentration', 'continues', 'cooperation', 'count', 'cracked', 'cramps', 'disposal', 'dissolve', 'doubt', 'dreams', 'dysmorphic', 'eight', 'emotional', 'fainted', 'fibroids', 'financial', 'found', 'fraustrated', 'government', 'groups', 'hypertension', 'impact', 'inadequacy', 'induce', 'injuries', 'installments', 'instructed', 'intolerance', 'itself', 'lactose', 'later', 'line', 'liquid', 'managed', 'menstrual', 'migraine', 'mindful', 'needles', 'occur']


In [18]:
STATS, COUNTS = {}, {}
def add(label, sentences, tokenize):
    s, c = vocab_stats(sentences, tokenize, label)
    STATS[label], COUNTS[label] = s, c

add("PHOENIX German (train)",       phx["train"]["translation"], ws_clean)
add("PHOENIX glosses (train)",      phx["train"]["orth"],        ws_clean)
add("TSL English, medical (train)", tr_en["text_en"],            ws_clean)
add("TSL Thai, all (train)",        trs["text_th_final"],        th_clean)
add("TSL Thai, cooking (train)",    trs[trs["batch"]=="batch1"]["text_th_final"], th_clean)
add("TSL Thai, medical (train)",    trs[trs["batch"]=="batch2"]["text_th_final"], th_clean)
for n in [1772, 2577]:
    add("PHOENIX German, %d sents" % n,
        phx["train"]["translation"].sample(n, random_state=42), ws_clean)

display(pd.DataFrame(STATS.values()).set_index("set").T)

set,PHOENIX German (train),PHOENIX glosses (train),"TSL English, medical (train)","TSL Thai, all (train)","TSL Thai, cooking (train)","TSL Thai, medical (train)","PHOENIX German, 1772 sents","PHOENIX German, 2577 sents"
sentences,7096.0,7096.0,1772.0,2577.0,805.0,1772.0,1772.0,2577.0
tokens,99081.0,56969.0,14448.0,48495.0,23951.0,24544.0,24506.0,35653.0
unique words,2887.0,953.0,2017.0,2318.0,1623.0,1502.0,1669.0,1958.0
avg times a word appears,34.3,59.8,7.2,20.9,14.8,16.3,14.7,18.2
words seen once %,37.3,26.7,45.0,36.9,40.4,35.8,39.6,39.8
words seen <5 times %,60.9,48.7,77.7,63.7,67.6,63.4,64.7,63.3
text covered by words seen >=5x %,97.1,98.6,81.7,94.8,92.4,93.2,92.7,94.3
words per sentence,14.0,8.0,8.2,18.8,29.8,13.9,13.8,13.8


In [19]:
# keep the whole Thai Unicode block (including combining marks); strip only real punctuation
PUNCT_TH = re.compile(r"[^\u0E00-\u0E7F\w\s]", flags=re.UNICODE)

def th_clean(s):
    s = unicodedata.normalize("NFC", str(s))
    s = PUNCT_TH.sub(" ", s)
    return [t for t in word_tokenize(s, engine="newmm") if t.strip()]

# proof it no longer damages the text
t = "คุณกระหายน้ำตลอดเวลาแม้จะดื่มน้ำแล้วไหม"
print("intact:", PUNCT_TH.sub(" ", t) == t)
print(th_clean(t))

intact: True
['คุณ', 'กระหายน้ำ', 'ตลอดเวลา', 'แม้', 'จะ', 'ดื่ม', 'น้ำ', 'แล้ว', 'ไหม']


In [20]:
add("TSL Thai, all (train)",     trs["text_th_final"], th_clean)
add("TSL Thai, cooking (train)", trs[trs["batch"]=="batch1"]["text_th_final"], th_clean)
add("TSL Thai, medical (train)", trs[trs["batch"]=="batch2"]["text_th_final"], th_clean)
display(pd.DataFrame(STATS.values()).set_index("set").T)

orows2 = [
    oov(phx["train"]["translation"], phx["test"]["translation"], ws_clean, "PHOENIX German"),
    oov(phx["train"]["orth"],        phx["test"]["orth"],        ws_clean, "PHOENIX glosses"),
    oov(tr_en["text_en"],            te_en["text_en"],           ws_clean, "TSL English (medical)"),
    oov(trs["text_th_final"],        te_sp["text_th_final"],     th_clean, "TSL Thai (all)"),
    oov(trs[trs["batch"]=="batch1"]["text_th_final"],
        te_sp[te_sp["batch"]=="batch1"]["text_th_final"], th_clean, "TSL Thai (cooking)"),
    oov(trs[trs["batch"]=="batch2"]["text_th_final"],
        te_sp[te_sp["batch"]=="batch2"]["text_th_final"], th_clean, "TSL Thai (medical)"),
]
display(pd.DataFrame(orows2).set_index("set").T)

set,PHOENIX German (train),PHOENIX glosses (train),"TSL English, medical (train)","TSL Thai, all (train)","TSL Thai, cooking (train)","TSL Thai, medical (train)","PHOENIX German, 1772 sents","PHOENIX German, 2577 sents"
sentences,7096.0,7096.0,1772.0,2577.0,805.0,1772.0,1772.0,2577.0
tokens,99081.0,56969.0,14448.0,29590.0,15308.0,14282.0,24506.0,35653.0
unique words,2887.0,953.0,2017.0,3403.0,2167.0,1916.0,1669.0,1958.0
avg times a word appears,34.3,59.8,7.2,8.7,7.1,7.5,14.7,18.2
words seen once %,37.3,26.7,45.0,43.6,47.2,44.5,39.6,39.8
words seen <5 times %,60.9,48.7,77.7,73.0,76.5,75.2,64.7,63.3
text covered by words seen >=5x %,97.1,98.6,81.7,86.0,82.3,83.1,92.7,94.3
words per sentence,14.0,8.0,8.2,11.5,19.0,8.1,13.8,13.8


set,PHOENIX German,PHOENIX glosses,TSL English (medical),TSL Thai (all),TSL Thai (cooking),TSL Thai (medical)
train vocabulary,2887.0,953.0,2017.0,3403.0,2167.0,1916.0
test tokens,7816.0,4399.0,1967.0,3758.0,1851.0,1907.0
unseen tokens %,0.8,0.3,5.6,5.0,8.3,4.7
unseen distinct words,59.0,12.0,103.0,178.0,141.0,86.0
test sentences with >=1 unseen word %,7.6,1.6,35.9,36.2,71.8,29.5


In [21]:
vocab_tbl = pd.DataFrame(STATS.values())
oov_tbl   = pd.DataFrame(orows2)
vocab_tbl.to_csv(W / "reports/vocab_comparison.csv", index=False)
oov_tbl.to_csv(W / "reports/oov_comparison.csv", index=False)
print(vocab_tbl.to_string(index=False))
print()
print(oov_tbl.to_string(index=False))
print("\nsaved to reports/vocab_comparison.csv and reports/oov_comparison.csv")

                         set  sentences  tokens  unique words  avg times a word appears  words seen once %  words seen <5 times %  text covered by words seen >=5x %  words per sentence
      PHOENIX German (train)       7096   99081          2887                      34.3               37.3                   60.9                               97.1                14.0
     PHOENIX glosses (train)       7096   56969           953                      59.8               26.7                   48.7                               98.6                 8.0
TSL English, medical (train)       1772   14448          2017                       7.2               45.0                   77.7                               81.7                 8.2
       TSL Thai, all (train)       2577   29590          3403                       8.7               43.6                   73.0                               86.0                11.5
   TSL Thai, cooking (train)        805   15308          2167              

# PHOENIX-2014T — vocabulary in plain numbers

**German sentences (training set, 7,096 sentences)**

| | number | meaning |
|---|---|---|
| total words written | 99,081 | counting every repeat |
| **unique words** | **2,887** | how many *different* words exist |
| **avg repeats per word** | **34x** | 99,081 / 2,887 — each word appears ~34 times |

**Glosses (the sign labels)**

| | number |
|---|---|
| total glosses | 56,969 |
| **unique glosses** | **953** |
| **avg repeats per gloss** | **60x** |

## Example

The word `regen` (rain) appears **838** times in the German sentences.
The gloss `REGEN` appears **2,149** times.

Most frequent German words: und, im, es, der, am, in, die, auch, grad, bis,
den, morgen, nacht, noch, an, heute, regen, sich, wolken, ein.

## Why it matters

PHOENIX is weather forecasts, so the same small set of words comes back again and again.
The model sees each sign dozens of times and can learn it well.

Our data: 2,017 unique English words but each appears only **7 times**.
Same vocabulary size, far less practice per word.

# PHOENIX vs TSL — Vocabulary Check

**Question:** our chrF is ~21 Is our method weak,
or is our task harder?

**Method:** counted word repetition in PHOENIX-2014T (7,096 train sentences, already on
the server) and in our data. A model can only learn a sign it has seen several times.

## Results

| | PHOENIX German | TSL English (medical) | TSL Thai (cooking) |
|---|---|---|---|
| sentences | 7,096 | 1,772 | 805 |
| unique words | 2,887 | 2,017 | 2,167 |
| **avg times a word appears** | **34×** | **7×** | **7×** |
| text made of words seen >=5x | 97% | 82% | 82% |
| **test sentences with an unseen word** | **7.6%** | **35.9%** | **71.8%** |

**Size control.** Taking a random 1,772 PHOENIX sentences (same count as ours), PHOENIX
still repeats each word **14.7x** vs our **7.2x**, coverage 93% vs 82%. Stable across
5 random draws. So about half the gap is corpus size, half is domain breadth.

## What it means

1. PHOENIX is weather forecasts — the same words return constantly (the gloss `REGEN`
   appears 2,149 times). Our data spans cooking plus 14 medical topics.
2. **One in three of our test sentences contains a word the model never saw** (PHOENIX:
   one in thirteen). Those sentences cannot be fully correct, whatever the architecture.
3. Cooking is the extreme case: 71.8% of its test sentences have an unseen word, and its
   sentences are 19 words long vs medical's 8 — which explains chrF 15.6 vs 25.5.

**Conclusion:** our scores and PHOENIX's are not measuring the same difficulty.
**Next data collection should prioritise more clips per sentence type over more types.** example - 
"Take this medicine after meals." / "Take this medicine with water." / "Don't stop taking this medicine."

## Caveats

- Thai tokenised with `pythainlp` (newmm), German/English by whitespace — so
  English-vs-German is the main comparison, Thai is supporting evidence.
- Two bugs found and fixed: punctuation attached to English words inflated our OOV rate
  (10.6% -> 5.6%); the first cleaning pass stripped Thai tone marks and was redone.
  Figures above are corrected.
- This explains why the task is hard; it does not excuse the model. The 46%-of-ceiling
  result stands separately.

Files: `reports/vocab_comparison.csv`, `reports/oov_comparison.csv`,
notebook `check_phoenixT.ipynb`.

# Why our scores are lower than PHOENIX — a vocabulary check

**Question.** Our chrF is ~21. Is our method weak, or is our task harder?

**Note.** SONAR-SLT reports BLEU / BLEURT / ROUGE, not chrF, and word-BLEU is unusable
on Thai (no word spaces). So we cannot compare scores directly — we compared the two
*datasets* instead. (Their BLEU 22.01 and our chrF 22.1 are different metrics; never
compare them.)

**Method.** A model can only learn a sign it has seen several times. So we counted, for
each dataset, how often each word repeats and how many test words were never seen in
training. PHOENIX-2014T was already on the server.

## Results

| | PHOENIX German | TSL English (medical) | TSL Thai (cooking) |
|---|---|---|---|
| sentences | 7,096 | 1,772 | 805 |
| unique words | 2,887 | 2,017 | 2,167 |
| **avg times a word appears** | **34x** | **7x** | **7x** |
| text made of words seen >=5x | 97% | 82% | 82% |
| **test sentences containing an unseen word** | **7.6%** | **35.9%** | **71.8%** |

**Size control.** With a random 1,772 PHOENIX sentences (same count as ours), PHOENIX
still repeats each word 14.7x vs our 7.2x. So roughly half the gap is corpus size, half
is domain breadth.

## What it means

PHOENIX is weather forecasts, so the same words return constantly (`regen` 838 times,
the gloss `REGEN` 2,149 times). Our data covers cooking plus 14 medical topics, so each
word appears about 7 times.

**One in three of our test sentences contains a word the model never saw**, against one
in thirteen for PHOENIX. Those sentences cannot be fully correct, whatever the
architecture. Cooking is the worst case at 71.8%, with 19-word sentences vs medical's 8
— which accounts for its chrF of 15.6 vs medical's 25.5.

## Recommendation for the next recording round

**Write the sentence list so the same words keep coming back**, instead of every
sentence introducing new ones.

> "Take this medicine after meals."
> "Take this medicine with water."
> "Don't stop taking this medicine."

Three different sentences — but the sign for *medicine* is seen three times. PHOENIX
gets this for free: 7,096 different sentences, all about weather.

Filming the same sentence with different signers also helps. Filming the identical
sentence repeatedly with one signer helps least.

A 10-minute check before filming: tokenise the planned sentence list and compute
average repeats per word. Below ~5 means the list is spread too thin.

## Caveats

- Thai tokenised with `pythainlp` (newmm), German/English by whitespace — so
  English-vs-German is the main comparison, Thai is supporting evidence.
- Two bugs found and fixed: punctuation glued to English words inflated our OOV rate
  (10.6% -> 5.6%), and the first cleaning pass stripped Thai tone marks. Figures above
  are the corrected ones.
- This describes task difficulty; it does not excuse the model. Our result of 46% of the
  achievable SONAR ceiling stands separately.
- To compare with the paper at all we need **BLEURT** (paper: 0.545 on PHOENIX-2014T).
  That is the next evaluation step.

Files: `reports/vocab_comparison.csv`, `reports/oov_comparison.csv`,
notebook `check_phoenixT.ipynb`.